# Phase 1: Exploratory Data Analysis (EDA) & Sampling

Because the Criteo dataset is 13 Million rows, we will use **Polars** to handle it quickly without crashing your CPU's RAM.

First, make sure the name of the file below matches the file you just downloaded into your `data/` folder!

In [ ]:
import polars as pl

# 1. Define the path. UPDATE THIS if your downloaded file is named differently! 
data_path = "../data/criteo-uplift-v2.1.csv"

# 2. Lazily scan the massive dataset (Runs instantly, bypasses RAM)
lazy_df = pl.scan_csv(data_path)

print("Scanner connected to hard drive successfully!")

### Extracting our 500,000 Row Working Sample
To build our baseline models perfectly, we will extract a random sample. Since the raw dataset was sorted with Treatment users at the top, we use a brilliant lazy math trick to pull every 26th row off the hard drive (randomizing perfectly across 13 Million rows) without ever crashing the RAM!

In [ ]:
# 3. Extract ~500,000 randomly distributed rows into RAM safely
sample_df = lazy_df.with_row_index("id").filter(pl.col("id") % 26 == 0).drop("id").collect()

# 4. Let's look at the exact balance of the Treatment vs Control group
print("\nTREATMENT VS CONTROL SPLIT:")
print(sample_df['treatment'].value_counts())

# 5. Let's look at the baseline Organic Conversion Rate (where treatment = 0)
control_group = sample_df.filter(pl.col('treatment') == 0)
print(f"\nControl Group Conversion Rate: {control_group['conversion'].mean() * 100:.2f}%")

# 6. Let's look at the Treatment Conversion Rate (where treatment = 1)
treatment_group = sample_df.filter(pl.col('treatment') == 1)
print(f"Treatment Group Conversion Rate: {treatment_group['conversion'].mean() * 100:.2f}%")

### Save the Data for Phase 2
If everything above looks correct and the code ran successfully, we will save this perfectly sized sample back to your `data` folder. We will use this file for the rest of the project!

In [ ]:
sample_path = "../data/sampled_uplift.csv"
sample_df.write_csv(sample_path)
print(f"Sample successfully saved to {sample_path}!")